Tile formation

In [ ]:
import os
import cv2
import numpy as np
from shapely.geometry import Polygon, box
from tqdm import tqdm
import json
from collections import defaultdict

# =========================================================
# CONFIGURATION
# =========================================================

INPUT_IMAGES_DIR = r"D:\CCRIPT AGENCY\Neptune_plumbing\train\images"
INPUT_LABELS_DIR = r"D:\CCRIPT AGENCY\Neptune_plumbing\train\labels"

OUTPUT_BASE_DIR = r"D:\CCRIPT AGENCY\Neptune_plumbing\final_tiled_coco"
OUTPUT_IMAGES_DIR = os.path.join(OUTPUT_BASE_DIR, "images")
COCO_JSON_PATH = os.path.join(OUTPUT_BASE_DIR, "annotations.json")

TILE_SIZE = 2048
OVERLAP_PERCENT = 0.40
STRIDE = int(TILE_SIZE * (1 - OVERLAP_PERCENT))

COVERAGE_THRESHOLD = 0.65
MIN_PIXEL_AREA = 80

NUM_CLASSES = 30
NOISE_CLASS_IDS = []   # keep empty unless needed

# COCO dict structure
coco = {
    "images": [],
    "annotations": [],
    "categories": [
        {"id": i, "name": f"class_{i}"} for i in range(NUM_CLASSES)
    ]
}

image_id_counter = 1
annotation_id_counter = 1

os.makedirs(OUTPUT_IMAGES_DIR, exist_ok=True)

def parse_obb_label(label_path, img_w, img_h):
    objects = []
    if not os.path.exists(label_path):
        return objects
    with open(label_path, "r") as f:
        lines = f.readlines()
    for line in lines:
        parts = line.strip().split()
        if len(parts) != 9:
            continue
        cls_id = int(parts[0])
        if cls_id < 0 or cls_id >= NUM_CLASSES:
            continue
        if cls_id in NOISE_CLASS_IDS:
            continue
        coords = list(map(float, parts[1:]))
        pts = []
        for i in range(0, 8, 2):
            x = coords[i] * img_w
            y = coords[i + 1] * img_h
            pts.append((x, y))
        poly = Polygon(pts)
        if not poly.is_valid or poly.area == 0:
            continue
        objects.append({"class": cls_id, "poly": poly, "points": pts})
    return objects

def polygon_in_tile(obj_poly, tile_poly):
    inter = obj_poly.intersection(tile_poly)
    if inter.is_empty:
        return None, 0.0, 0.0
    if inter.geom_type == "MultiPolygon":
        inter_poly = max(inter.geoms, key=lambda g: g.area)
    elif inter.geom_type == "Polygon":
        inter_poly = inter
    else:
        return None, 0.0, 0.0
    coverage = inter_poly.area / obj_poly.area
    return inter_poly, coverage, inter_poly.area

def coco_bbox_from_poly(poly):
    minx, miny, maxx, maxy = poly.bounds
    return [minx, miny, maxx - minx, maxy - miny]

def save_tile(tile_img, objects, tile_x, tile_y, prefix):
    global image_id_counter, annotation_id_counter
    tile_poly = box(tile_x, tile_y, tile_x + TILE_SIZE, tile_y + TILE_SIZE)
    valid = []
    annots = []
    for obj in objects:
        inter_poly, coverage, area = polygon_in_tile(obj["poly"], tile_poly)
        if inter_poly is None:
            continue
        if coverage < COVERAGE_THRESHOLD:
            continue
        if area < MIN_PIXEL_AREA:
            continue
        # Shift polygon to tile coordinates
        shifted = [(x - tile_x, y - tile_y) for x, y in inter_poly.exterior.coords[:-1]]
        # COCO expects segmentation as a list of [x1, y1, x2, y2, ...]
        segmentation = [v for xy in shifted for v in xy]
        bbox = coco_bbox_from_poly(Polygon(shifted))
        annots.append({
            "category_id": obj["class"],
            "segmentation": [segmentation],
            "bbox": bbox,
            "area": Polygon(shifted).area,
            "iscrowd": 0
        })
    if len(annots) == 0:
        return
    name = f"{prefix}_{image_id_counter:06d}.jpg"
    cv2.imwrite(os.path.join(OUTPUT_IMAGES_DIR, name), tile_img)
    coco["images"].append({
        "id": image_id_counter,
        "file_name": name,
        "width": TILE_SIZE,
        "height": TILE_SIZE
    })
    for a in annots:
        a["image_id"] = image_id_counter
        a["id"] = annotation_id_counter
        coco["annotations"].append(a)
        annotation_id_counter += 1
    image_id_counter += 1

def process_images():
    image_files = [
        f for f in os.listdir(INPUT_IMAGES_DIR)
        if f.lower().endswith((".jpg", ".png", ".jpeg"))
    ]
    print(f"Found {len(image_files)} images")
    print(f"TILE SIZE = {TILE_SIZE}, STRIDE = {STRIDE}")
    for img_file in tqdm(image_files):
        img_path = os.path.join(INPUT_IMAGES_DIR, img_file)
        lbl_path = os.path.join(INPUT_LABELS_DIR, os.path.splitext(img_file)[0] + ".txt")
        img = cv2.imread(img_path)
        if img is None:
            continue
        h, w = img.shape[:2]
        objects = parse_obb_label(lbl_path, w, h)
        xs = list(range(0, max(1, w - TILE_SIZE + 1), STRIDE))
        ys = list(range(0, max(1, h - TILE_SIZE + 1), STRIDE))
        if xs[-1] != w - TILE_SIZE:
            xs.append(w - TILE_SIZE)
        if ys[-1] != h - TILE_SIZE:
            ys.append(h - TILE_SIZE)
        for y in ys:
            for x in xs:
                tile = img[y:y + TILE_SIZE, x:x + TILE_SIZE]
                if tile.shape[0] != TILE_SIZE or tile.shape[1] != TILE_SIZE:
                    continue
                save_tile(tile, objects, x, y, os.path.splitext(img_file)[0])
    print(f"\n✅ Tiling complete. Images: {len(coco['images'])}, Annotations: {len(coco['annotations'])}")
    with open(COCO_JSON_PATH, "w") as f:
        json.dump(coco, f, indent=2)
    print(f"COCO annotations saved to {COCO_JSON_PATH}")

if __name__ == "__main__":
    process_images()


Found 12 images
TILE SIZE = 2048, STRIDE = 1228


100%|██████████| 12/12 [00:21<00:00,  1.77s/it]


✅ Positive tiles: 124
✅ Background tiles: 668


Visualizations

In [6]:
import os
import cv2
import yaml
import numpy as np
from pathlib import Path
from tqdm import tqdm

# ---------------- CONFIG ----------------
DATASET_DIR = r"D:\CCRIPT AGENCY\Neptune_plumbing\train"
IMAGES_DIR = os.path.join(DATASET_DIR, "images")
LABELS_DIR = os.path.join(DATASET_DIR, "labels")
YAML_PATH = os.path.join(DATASET_DIR, "data.yaml")

OUTPUT_DIR = r"D:\CCRIPT AGENCY\Neptune_plumbing\obb_background_synthetic_visual_check_balancedv4"
MAX_IMAGES = 50        # number of tiles to visualize
FONT_SCALE = 0.4
THICKNESS = 2
# --------------------------------------


# Load class names
with open(YAML_PATH, "r") as f:
    CLASS_NAMES = yaml.safe_load(f)["names"]


os.makedirs(OUTPUT_DIR, exist_ok=True)


def draw_obb(img, cls_id, points):
    """
    points: list of (x, y) pixel coordinates
    """
    pts = np.array(points, dtype=np.int32)

    # Draw polygon
    cv2.polylines(img, [pts], isClosed=True, color=(0, 255, 0), thickness=THICKNESS)

    # Draw class name near first point
    label = CLASS_NAMES[cls_id]
    x, y = pts[0]

    cv2.putText(
        img,
        label,
        (x, max(0, y - 5)),
        cv2.FONT_HERSHEY_SIMPLEX,
        FONT_SCALE,
        (0, 255, 0),
        1,
        cv2.LINE_AA
    )


def visualize():
    image_files = sorted([
        f for f in os.listdir(IMAGES_DIR)
        if f.lower().endswith((".jpg", ".png"))
    ])[:MAX_IMAGES]

    print(f"Visualizing {len(image_files)} tiles...")

    for img_file in tqdm(image_files):
        img_path = os.path.join(IMAGES_DIR, img_file)
        lbl_path = os.path.join(LABELS_DIR, Path(img_file).stem + ".txt")

        img = cv2.imread(img_path)
        if img is None:
            continue

        h, w = img.shape[:2]

        if os.path.exists(lbl_path):
            with open(lbl_path, "r") as f:
                lines = f.readlines()

            for line in lines:
                parts = line.strip().split()
                if len(parts) != 9:
                    continue

                cls_id = int(parts[0])
                coords = list(map(float, parts[1:]))

                points = []
                for i in range(0, 8, 2):
                    x = int(coords[i] * w)
                    y = int(coords[i + 1] * h)
                    points.append((x, y))

                draw_obb(img, cls_id, points)

        out_path = os.path.join(OUTPUT_DIR, img_file)
        cv2.imwrite(out_path, img)

    print(f"✅ Visualization saved to: {OUTPUT_DIR}")


if __name__ == "__main__":
    visualize()

Visualizing 12 tiles...


100%|██████████| 12/12 [00:07<00:00,  1.56it/s]

✅ Visualization saved to: D:\CCRIPT AGENCY\Neptune_plumbing\obb_background_synthetic_visual_check_balancedv4


Adding symbols to tiles with no annotations

In [ ]:
import cv2
import random
import numpy as np
from shapely.geometry import Polygon, box
from pathlib import Path
from collections import Counter, defaultdict
from tqdm import tqdm
import json
import os

# ================= CONFIG =================
DATASET_DIR = r"D:\CCRIPT AGENCY\Neptune_plumbing\final_tiled_coco"
IMAGES_DIR = Path(DATASET_DIR) / "images"
COCO_JSON_PATH = Path(DATASET_DIR) / "annotations.json"
AUG_COCO_JSON_PATH = Path(DATASET_DIR) / "aug_annotations.json"

SYMBOL_CROPS_DIR = Path(DATASET_DIR) / "symbol_crops"

MIN_SYMBOLS = 30
MAX_SYMBOLS = 40
MAX_ATTEMPTS = 30
MAX_CLASS_FRACTION = 0.3   # max 30% symbols of one class per tile

# ================= LOAD COCO =================
with open(COCO_JSON_PATH, "r") as f:
    coco = json.load(f)

NUM_CLASSES = len(coco["categories"])

# ================= LOAD SYMBOL CROPS =================
SYMBOL_LIBRARY = defaultdict(list)
if SYMBOL_CROPS_DIR.exists():
    for class_dir in SYMBOL_CROPS_DIR.glob("class_*"):
        cid = int(class_dir.name.split("_")[1])
        for img_path in class_dir.glob("*.png"):
            SYMBOL_LIBRARY[cid].append(img_path)

# ================= GLOBAL CLASS COUNTS =================
def global_class_counts():
    c = Counter()
    for ann in coco["annotations"]:
        c[ann["category_id"]] += 1
    return c

GLOBAL_COUNTS = global_class_counts()
median = np.median(list(GLOBAL_COUNTS.values()))
RARE_CLASSES = [c for c, v in GLOBAL_COUNTS.items() if v < 0.4 * median]

# ================= AUGMENTATION =================
aug_coco = {
    "images": [],
    "annotations": [],
    "categories": coco["categories"]
}
image_id_counter = 1
annotation_id_counter = 1

for img_info in tqdm(coco["images"]):
    img_path = IMAGES_DIR / img_info["file_name"]
    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]
    anns = [a for a in coco["annotations"] if a["image_id"] == img_info["id"]]
    polys = [(a["category_id"], Polygon(np.array(a["segmentation"][0]).reshape(-1,2))) for a in anns]
    label_lines = anns.copy()
    per_class = Counter(cid for cid, _ in polys)
    if len(polys) >= MIN_SYMBOLS:
        # Just copy
        new_img_name = f"aug_{image_id_counter:07d}.jpg"
        cv2.imwrite(str(IMAGES_DIR / new_img_name), img)
        aug_coco["images"].append({
            "id": image_id_counter,
            "file_name": new_img_name,
            "width": w,
            "height": h
        })
        for a in anns:
            a2 = a.copy()
            a2["image_id"] = image_id_counter
            a2["id"] = annotation_id_counter
            aug_coco["annotations"].append(a2)
            annotation_id_counter += 1
        image_id_counter += 1
        continue
    target = random.randint(MIN_SYMBOLS, MAX_SYMBOLS)
    while len(polys) < target:
        # ---- pick class (rare first) ----
        if random.random() < 0.7 and RARE_CLASSES:
            cid = random.choice(RARE_CLASSES)
        else:
            cid = random.choice(list(SYMBOL_LIBRARY.keys()))
        if per_class[cid] >= MAX_CLASS_FRACTION * target:
            continue
        if cid not in SYMBOL_LIBRARY:
            continue
        crop = cv2.imread(str(random.choice(SYMBOL_LIBRARY[cid])))
        ch, cw = crop.shape[:2]
        if cw >= w or ch >= h:
            continue
        placed = False
        for _ in range(MAX_ATTEMPTS):
            x = random.randint(0, w - cw - 1)
            y = random.randint(0, h - ch - 1)
            new_poly = Polygon([(x,y),(x+cw,y),(x+cw,y+ch),(x,y+ch)])
            if any(new_poly.intersects(p) for _, p in polys):
                continue
            img[y:y+ch, x:x+cw] = crop
            polys.append((cid, new_poly))
            per_class[cid] += 1
            segmentation = [v for xy in new_poly.exterior.coords[:-1] for v in xy]
            bbox = [x, y, cw, ch]
            aug_coco["annotations"].append({
                "id": annotation_id_counter,
                "image_id": image_id_counter,
                "category_id": cid,
                "segmentation": [segmentation],
                "bbox": bbox,
                "area": new_poly.area,
                "iscrowd": 0
            })
            annotation_id_counter += 1
            placed = True
            break
        if not placed:
            break
    new_img_name = f"aug_{image_id_counter:07d}.jpg"
    cv2.imwrite(str(IMAGES_DIR / new_img_name), img)
    aug_coco["images"].append({
        "id": image_id_counter,
        "file_name": new_img_name,
        "width": w,
        "height": h
    })
    image_id_counter += 1

with open(AUG_COCO_JSON_PATH, "w") as f:
    json.dump(aug_coco, f, indent=2)
print(f"✅ Augmented COCO saved to {AUG_COCO_JSON_PATH}")


100%|██████████| 124/124 [00:17<00:00,  7.06it/s]

✅ Balanced paste-from-crops augmentation completed


Total class counts

In [9]:
import os
from collections import defaultdict

# Path to your labels folder
LABELS_DIR = r"D:\CCRIPT AGENCY\Neptune_plumbing\final_tiled_obb_balancedv1\labels"   # change if needed

class_counts = defaultdict(int)
total_labels = 0

for file in os.listdir(LABELS_DIR):
    if not file.endswith(".txt"):
        continue

    path = os.path.join(LABELS_DIR, file)

    with open(path, "r") as f:
        lines = f.readlines()

    for line in lines:
        parts = line.strip().split()
        if len(parts) == 0:
            continue
        
        cls_id = int(parts[0])     # first value is class_id
        class_counts[cls_id] += 1
        total_labels += 1

# Print results
print("\n===== CLASS COUNTS =====")
for cls_id, count in sorted(class_counts.items()):
    print(f"Class {cls_id}: {count}")

print("\nTotal labels:", total_labels)
print("========================\n")



===== CLASS COUNTS =====
Class 0: 66
Class 2: 95
Class 3: 97
Class 4: 91
Class 5: 184
Class 6: 80
Class 7: 68
Class 8: 104
Class 9: 126
Class 10: 170
Class 11: 69
Class 12: 61
Class 13: 1335
Class 14: 63
Class 15: 91
Class 16: 96
Class 17: 80
Class 18: 77
Class 19: 87
Class 20: 106
Class 21: 73
Class 22: 81
Class 23: 74
Class 24: 79
Class 25: 124
Class 26: 25
Class 27: 78
Class 28: 85
Class 29: 88

Total labels: 3853



Synthetic Data Generation

In [ ]:
import cv2
import random
import numpy as np
from shapely.geometry import Polygon
from pathlib import Path
from collections import Counter, defaultdict
from tqdm import tqdm
import json
import os

# =====================================================
# CONFIG
# =====================================================
DATASET_DIR = r"D:\CCRIPT AGENCY\Neptune_plumbing\final_tiled_coco"
BG_IMAGES_DIR = Path(DATASET_DIR) / "backgrounds/images"
SYMBOL_CROPS_DIR = Path(DATASET_DIR) / "symbol_crops"
AUG_COCO_JSON_PATH = Path(DATASET_DIR) / "aug_annotations.json"
SYN_COCO_JSON_PATH = Path(DATASET_DIR) / "syn_annotations.json"
IMAGES_DIR = Path(DATASET_DIR) / "images"

MIN_SYMBOLS = 40
MAX_SYMBOLS = 60
MAX_ATTEMPTS = 30
MAX_CLASS_FRACTION = 0.35

# =====================================================
# LOAD COCO
# =====================================================
with open(AUG_COCO_JSON_PATH, "r") as f:
    coco = json.load(f)

NUM_CLASSES = len(coco["categories"])

# =====================================================
# LOAD SYMBOL CROPS
# =====================================================
SYMBOL_LIBRARY = defaultdict(list)
if SYMBOL_CROPS_DIR.exists():
    for class_dir in SYMBOL_CROPS_DIR.glob("class_*"):
        cid = int(class_dir.name.split("_")[1])
        for img_path in class_dir.glob("*.png"):
            SYMBOL_LIBRARY[cid].append(img_path)

# =====================================================
# LOAD REAL CLASS COUNTS
# =====================================================
def load_real_class_counts():
    counts = Counter()
    for ann in coco["annotations"]:
        counts[ann["category_id"]] += 1
    return counts

REAL_COUNTS = load_real_class_counts()
median = np.median(list(REAL_COUNTS.values()))

DEFICIT = {}
for c, real_cnt in REAL_COUNTS.items():
    if real_cnt < 0.3 * median:
        target = int(0.65 * median)
    elif real_cnt < median:
        target = int(0.9 * median)
    else:
        target = real_cnt
    DEFICIT[c] = max(0, target - real_cnt)
DEFICIT_CLASSES = [c for c, d in DEFICIT.items() if d > 0]

# =====================================================
# SYNTHETIC GENERATION
# =====================================================
syn_coco = {
    "images": [],
    "annotations": [],
    "categories": coco["categories"]
}
image_id_counter = 1
annotation_id_counter = 1

for bg_path in tqdm(list(BG_IMAGES_DIR.glob("*.jpg"))):
    if all(v <= 0 for v in DEFICIT.values()):
        break
    img = cv2.imread(str(bg_path))
    h, w = img.shape[:2]
    polys = []
    labels = []
    per_class = Counter()
    target = random.randint(MIN_SYMBOLS, MAX_SYMBOLS)
    MAX_TOTAL_ITERS = target * 20
    iters = 0
    while len(polys) < target and iters < MAX_TOTAL_ITERS:
        iters += 1
        eligible = [
            c for c in DEFICIT_CLASSES
            if DEFICIT[c] > 0 and per_class[c] < MAX_CLASS_FRACTION * target
        ]
        if not eligible:
            break
        weights = np.array([DEFICIT[c] for c in eligible], dtype=float)
        weights /= weights.sum()
        cid = random.choices(eligible, weights=weights, k=1)[0]
        crop = cv2.imread(str(random.choice(SYMBOL_LIBRARY[cid])))
        ch, cw = crop.shape[:2]
        if cw >= w or ch >= h:
            continue
        placed = False
        for _ in range(MAX_ATTEMPTS):
            x = random.randint(0, w - cw - 1)
            y = random.randint(0, h - ch - 1)
            poly = Polygon([
                (x, y),
                (x + cw, y),
                (x + cw, y + ch),
                (x, y + ch)
            ])
            if any(poly.intersects(p) for _, p in polys):
                continue
            img[y:y+ch, x:x+cw] = crop
            polys.append((cid, poly))
            per_class[cid] += 1
            DEFICIT[cid] -= 1
            segmentation = [v for xy in poly.exterior.coords[:-1] for v in xy]
            bbox = [x, y, cw, ch]
            syn_coco["annotations"].append({
                "id": annotation_id_counter,
                "image_id": image_id_counter,
                "category_id": cid,
                "segmentation": [segmentation],
                "bbox": bbox,
                "area": poly.area,
                "iscrowd": 0
            })
            annotation_id_counter += 1
            placed = True
            break
        if not placed:
            continue
    if not polys:
        continue
    new_img_name = f"syn_{image_id_counter:07d}.jpg"
    cv2.imwrite(str(IMAGES_DIR / new_img_name), img)
    syn_coco["images"].append({
        "id": image_id_counter,
        "file_name": new_img_name,
        "width": w,
        "height": h
    })
    image_id_counter += 1

with open(SYN_COCO_JSON_PATH, "w") as f:
    json.dump(syn_coco, f, indent=2)
print(f"\n✅ Synthetic COCO saved to {SYN_COCO_JSON_PATH}")



📊 Median of real data: 85

📉 Deficit classes (capped):
Class  0 | real=   66 | target=   76 | deficit=   10
Class  7 | real=   68 | target=   76 | deficit=    8
Class 11 | real=   69 | target=   76 | deficit=    7
Class 12 | real=   61 | target=   76 | deficit=   15
Class 14 | real=   63 | target=   76 | deficit=   13
Class 21 | real=   73 | target=   76 | deficit=    3
Class 23 | real=   74 | target=   76 | deficit=    2
Class 26 | real=   25 | target=   55 | deficit=   30

🚀 Generating capped-deficit background synthetic tiles...


  0%|          | 0/668 [00:00<?, ?it/s]

  1%|          | 5/668 [00:45<1:41:11,  9.16s/it]


✅ Background synthetic generation completed

📊 Remaining deficits:
Class  0: remaining deficit 0
Class  2: remaining deficit 0
Class  3: remaining deficit 0
Class  4: remaining deficit 0
Class  5: remaining deficit 0
Class  6: remaining deficit 0
Class  7: remaining deficit 0
Class  8: remaining deficit 0
Class  9: remaining deficit 0
Class 10: remaining deficit 0
Class 11: remaining deficit 0
Class 12: remaining deficit 0
Class 13: remaining deficit 0
Class 14: remaining deficit 0
Class 15: remaining deficit 0
Class 16: remaining deficit 0
Class 17: remaining deficit 0
Class 18: remaining deficit 0
Class 19: remaining deficit 0
Class 20: remaining deficit 0
Class 21: remaining deficit 0
Class 22: remaining deficit 0
Class 23: remaining deficit 0
Class 24: remaining deficit 0
Class 25: remaining deficit 0
Class 26: remaining deficit 0
Class 27: remaining deficit 0
Class 28: remaining deficit 0
Class 29: remaining deficit 0


Merging both datasets

In [ ]:
import json
import random
from tqdm import tqdm
import os
import shutil

# ================= CONFIG =================
AUG_COCO_JSON_PATH = r"D:\CCRIPT AGENCY\Neptune_plumbing\final_tiled_coco\aug_annotations.json"
SYN_COCO_JSON_PATH = r"D:\CCRIPT AGENCY\Neptune_plumbing\final_tiled_coco\syn_annotations.json"
MERGED_COCO_JSON_PATH = r"D:\CCRIPT AGENCY\Neptune_plumbing\final_tiled_coco\merged_annotations.json"
IMAGES_DIR = r"D:\CCRIPT AGENCY\Neptune_plumbing\final_tiled_coco\images"

MAX_SYN_RATIO = 1.0   # <= 100% synthetic relative to real

# ================= LOAD COCO =================
with open(AUG_COCO_JSON_PATH, "r") as f:
    aug_coco = json.load(f)
with open(SYN_COCO_JSON_PATH, "r") as f:
    syn_coco = json.load(f)

# ================= MERGE =================
real_imgs = aug_coco["images"]
real_anns = aug_coco["annotations"]
syn_imgs = syn_coco["images"]
syn_anns = syn_coco["annotations"]

max_syn = int(len(real_imgs) * MAX_SYN_RATIO)
if len(syn_imgs) > max_syn:
    syn_imgs = random.sample(syn_imgs, max_syn)
    syn_img_ids = set(i["id"] for i in syn_imgs)
    syn_anns = [a for a in syn_anns if a["image_id"] in syn_img_ids]

# Re-index image and annotation ids for merged COCO
offset_img = 0
merged_imgs = []
merged_anns = []
img_id_map = {}
ann_id = 1
for img in real_imgs + syn_imgs:
    new_img = img.copy()
    new_img["id"] = len(merged_imgs) + 1
    img_id_map[img["id"]] = new_img["id"]
    merged_imgs.append(new_img)
for ann in real_anns + syn_anns:
    new_ann = ann.copy()
    new_ann["id"] = ann_id
    new_ann["image_id"] = img_id_map[ann["image_id"]]
    merged_anns.append(new_ann)
    ann_id += 1

merged_coco = {
    "images": merged_imgs,
    "annotations": merged_anns,
    "categories": aug_coco["categories"]
}

with open(MERGED_COCO_JSON_PATH, "w") as f:
    json.dump(merged_coco, f, indent=2)
print(f"✅ Real + Synthetic datasets merged in COCO format: {MERGED_COCO_JSON_PATH}")


Real samples      : 124
Synthetic samples : 5 (capped)


100%|██████████| 129/129 [00:00<00:00, 436.38it/s]

✅ Real + Synthetic datasets merged (synthetic capped)
Total images: 129


Final class counts

In [13]:
import os
from collections import defaultdict

# Path to your labels folder
LABELS_DIR = r"D:\CCRIPT AGENCY\Neptune_plumbing\final_tiled_obb_merged\labels"   # change if needed

class_counts = defaultdict(int)
total_labels = 0

for file in os.listdir(LABELS_DIR):
    if not file.endswith(".txt"):
        continue

    path = os.path.join(LABELS_DIR, file)

    with open(path, "r") as f:
        lines = f.readlines()

    for line in lines:
        parts = line.strip().split()
        if len(parts) == 0:
            continue
        
        cls_id = int(parts[0])     # first value is class_id
        class_counts[cls_id] += 1
        total_labels += 1

# Print results
print("\n===== CLASS COUNTS =====")
for cls_id, count in sorted(class_counts.items()):
    print(f"Class {cls_id}: {count}")

print("\nTotal labels:", total_labels)
print("========================\n")


===== CLASS COUNTS =====
Class 0: 76
Class 2: 95
Class 3: 97
Class 4: 91
Class 5: 184
Class 6: 80
Class 7: 76
Class 8: 104
Class 9: 126
Class 10: 170
Class 11: 76
Class 12: 76
Class 13: 1335
Class 14: 76
Class 15: 91
Class 16: 96
Class 17: 80
Class 18: 77
Class 19: 87
Class 20: 106
Class 21: 76
Class 22: 81
Class 23: 76
Class 24: 79
Class 25: 124
Class 26: 55
Class 27: 78
Class 28: 85
Class 29: 88

Total labels: 3941



Splitting

In [ ]:
import json
import random
import os
import shutil
from tqdm import tqdm

# ----------------------------------------------
# CONFIG
# ----------------------------------------------
MERGED_COCO_JSON_PATH = r"D:\CCRIPT AGENCY\Neptune_plumbing\final_tiled_coco\merged_annotations.json"
IMAGES_DIR = r"D:\CCRIPT AGENCY\Neptune_plumbing\final_tiled_coco\images"
OUT_DIR = r"D:\CCRIPT AGENCY\Neptune_plumbing\final_tiled_coco\split"

TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10
RANDOM_SEED = 42

random.seed(RANDOM_SEED)

# ----------------------------------------------
# LOAD COCO
# ----------------------------------------------
with open(MERGED_COCO_JSON_PATH, "r") as f:
    coco = json.load(f)

images = coco["images"]
annotations = coco["annotations"]
categories = coco["categories"]

# ----------------------------------------------
# SPLIT
# ----------------------------------------------
random.shuffle(images)
total = len(images)
train_end = int(total * TRAIN_RATIO)
val_end = train_end + int(total * VAL_RATIO)
splits = {
    "train": images[:train_end],
    "val": images[train_end:val_end],
    "test": images[val_end:]
}

for s in splits:
    os.makedirs(os.path.join(OUT_DIR, s, "images"), exist_ok=True)
    os.makedirs(os.path.join(OUT_DIR, s, "annotations"), exist_ok=True)

# ----------------------------------------------
# COPY FILES & EXPORT COCO
# ----------------------------------------------
for split, imgs in splits.items():
    print(f"\nCopying {split} ({len(imgs)} samples)...")
    split_img_ids = set(i["id"] for i in imgs)
    split_anns = [a for a in annotations if a["image_id"] in split_img_ids]
    split_coco = {
        "images": imgs,
        "annotations": split_anns,
        "categories": categories
    }
    with open(os.path.join(OUT_DIR, split, "annotations", f"{split}.json"), "w") as f:
        json.dump(split_coco, f, indent=2)
    for img in tqdm(imgs):
        src = os.path.join(IMAGES_DIR, img["file_name"])
        dst = os.path.join(OUT_DIR, split, "images", img["file_name"])
        shutil.copy(src, dst)
print("\n✔ COCO dataset split complete.")


Valid samples: 129

Copying train (103 samples)...


  0%|          | 0/103 [00:00<?, ?it/s]

100%|██████████| 103/103 [00:00<00:00, 336.57it/s]



Copying val (12 samples)...


100%|██████████| 12/12 [00:00<00:00, 553.98it/s]



Copying test (14 samples)...


100%|██████████| 14/14 [00:00<00:00, 332.83it/s]


✔ Class-aware dataset split complete.


Optimizing tool selection...Here’s a detailed explanation of the logic and reasoning behind each step and cell in your notebook, focusing on why each step is necessary in the context of building a robust, balanced object detection dataset from large images with class imbalance and sparse annotations.

---

### 1. Tile formation (Cell 1-2)
**Logic:**  
- Large images are hard to process and train on directly.
- Tiling breaks them into manageable, fixed-size pieces (2048x2048) for efficient model training.
- Overlap (40%) ensures objects near tile edges are not lost and appear in multiple tiles, improving detection robustness.
- Extracting symbol crops at this stage enables later data augmentation and synthetic data generation.
- Background tiles (mostly empty) are separated for later use in synthetic data generation.

---

### 2. Visualizations (Cell 3-4)
**Logic:**  
- Visualizing tiles with drawn bounding boxes is a quality control step.
- Ensures that tiling and label transformation worked as intended.
- Helps spot errors in annotation conversion, tiling, or class mapping before proceeding to augmentation and training.

---

### 3. Adding symbols to tiles with no annotations (Cell 5-6)
**Logic:**  
- After tiling, many tiles may have few or no objects, especially for rare classes.
- Pasting symbol crops into sparse tiles increases object density, making the dataset more useful for training.
- Prioritizing rare classes (70% chance) helps balance the dataset, so the model doesn’t ignore underrepresented classes.
- Limiting the number of objects per class per tile (max 30%) ensures diversity and prevents overfitting to a single class.
- Overlap checks prevent unrealistic or ambiguous training data.

---

### 4. Total class counts (Cell 7-8)
**Logic:**  
- Counting class occurrences after augmentation quantifies the effect of balancing.
- Identifies if further balancing or synthetic generation is needed.
- Ensures that the dataset is not dominated by a few classes, which would bias the model.

---

### 5. Synthetic Data Generation (Cell 9-10)
**Logic:**  
- Even after augmentation, some classes may still be underrepresented.
- Synthetic data generation fills these “deficits” by pasting symbols onto background tiles.
- Uses a two-tier strategy: very rare classes get a bigger boost, rare classes get a moderate boost, common classes are left as-is.
- Sampling is weighted by deficit, so classes with the biggest shortfall are prioritized.
- Stops generating once all deficits are filled, making the process efficient and targeted.

---

### 6. Merging both datasets (Cell 11-12)
**Logic:**  
- Real (augmented) and synthetic data are merged into a single dataset for training.
- Synthetic data is capped (e.g., at 1:1 ratio) to prevent it from overwhelming real data, maintaining realism.
- Shuffling ensures that batches during training are diverse and not segregated by source (real/synthetic).

---

### 7. Final class counts (Cell 13-14)
**Logic:**  
- Re-counting classes after merging confirms that balancing and synthetic generation achieved the desired effect.
- Final check before splitting and training.

---

### 8. Splitting (Cell 15-16)
**Logic:**  
- Splitting into train/val/test sets is essential for unbiased model evaluation.
- 80/10/10 split is standard, providing enough data for training and reliable validation/testing.
- Random shuffling ensures splits are representative and not biased by order or source.
- Skipping empty labels ensures only useful data is included.

---

### 9. (Empty cell at end)
**Logic:**  
- No logic; just a placeholder or end-of-notebook artifact.

---

## **Summary of the Pipeline’s Logic**

- **Tiling**: Makes large images manageable and ensures all objects are seen.
- **Visualization**: Early error detection and quality control.
- **Augmentation (paste-from-crops)**: Increases object density and balances rare classes.
- **Class counting**: Monitors progress toward a balanced dataset.
- **Synthetic generation**: Fills remaining class deficits in a targeted, efficient way.
- **Merging**: Combines all data for a unified training set, with synthetic capped to preserve realism.
- **Final counting**: Confirms balance before splitting.
- **Splitting**: Prepares data for robust, unbiased model training and evaluation.

**Overall, every step is designed to maximize the diversity, balance, and quality of your object detection dataset, ensuring your model can learn all classes well and generalize to new data.**